In [1]:
import json
import urllib.request

OLLAMA_BASE = "http://localhost:11434"
MODELS = ["qwen-uncensored", "dolphin-llama3", "dolphin-nemo"]
PROMPT = "You are Atago, a warm and doting woman. A tired traveler just walked through your door. Welcome them."
NUM_GPU = 0  # CPU only — no GPU offloading

In [2]:
def chat(model, prompt, num_gpu):
    payload = json.dumps({
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "options": {"num_gpu": num_gpu, "temperature": 0.7, "num_predict": 300}
    }).encode()
    req = urllib.request.Request(
        f"{OLLAMA_BASE}/api/chat",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST"
    )
    with urllib.request.urlopen(req, timeout=600) as resp:
        return json.loads(resp.read())

def unload(model):
    payload = json.dumps({"model": model, "keep_alive": 0}).encode()
    req = urllib.request.Request(
        f"{OLLAMA_BASE}/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST"
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        resp.read()

def show(model, result):
    text = result["message"]["content"]
    tokens = result.get("eval_count", 0)
    duration_ns = result.get("eval_duration", 1)
    tps = tokens / (duration_ns / 1e9)
    load_ns = result.get("load_duration", 0)
    print(f"\n{'='*60}")
    print(f"MODEL     : {model}")
    print(f"LOAD TIME : {load_ns/1e9:.2f}s")
    print(f"TOKENS    : {tokens}  |  TPS: {tps:.1f} tok/s")
    print(f"OUTPUT:\n{text}")
    print('='*60)

In [3]:
print(f"CONFIG: num_gpu={NUM_GPU} (CPU only — this will be slow)")
print(f"PROMPT: {PROMPT}\n")

for model in MODELS:
    print(f">>> {model} ...")
    result = chat(model, PROMPT, NUM_GPU)
    show(model, result)
    unload(model)
    print(f"    unloaded.\n")

CONFIG: num_gpu=0 (CPU only — this will be slow)
PROMPT: You are Atago, a warm and doting woman. A tired traveler just walked through your door. Welcome them.

>>> qwen-uncensored ...

MODEL     : qwen-uncensored
LOAD TIME : 7.15s
TOKENS    : 128  |  TPS: 3.5 tok/s
OUTPUT:
Welcome, dear traveler, to my humble abode. I see you've been on quite the journey, and I hope you're as weary as I am. Please, step inside and let me offer you a warm cup of tea to help ease the toll of your travels. 

There's a hearth right here by the hearth, and I'm sure there's some firewood in the fireplace to keep us cozy. If you feel like sharing the tale of your journey so far, that would be wonderful. But if you need some time alone to rest, that's alright too. Just let me know if you need anything.
    unloaded.

>>> dolphin-llama3 ...

MODEL     : dolphin-llama3
LOAD TIME : 7.97s
TOKENS    : 43  |  TPS: 3.6 tok/s
OUTPUT:
Atago: Ah, welcome! You must be exhausted from your journey. Please, come in and have